In [79]:
import numpy as np
import pandas as pd
import optuna
import itertools
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset e Preprocessing
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.feature_selection import SelectFromModel
from sklearn.base import BaseEstimator, RegressorMixin

# Modelli
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor

In [80]:
#Importazione del modello
data_path = "world_happiness_report.csv"
df = pd.read_csv(data_path)

In [81]:
#info dei valori nulli
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     1231 non-null   int64  
 1   Country                        617 non-null    object 
 2   Region                         315 non-null    object 
 3   Happiness Rank                 315 non-null    float64
 4   Happiness Score                315 non-null    float64
 5   Standard Error                 158 non-null    float64
 6   Economy (GDP per Capita)       315 non-null    float64
 7   Family                         470 non-null    float64
 8   Health (Life Expectancy)       315 non-null    float64
 9   Freedom                        470 non-null    float64
 10  Trust (Government Corruption)  315 non-null    float64
 11  Generosity                     1084 non-null   float64
 12  Dystopia Residual              315 non-null    f

**Analisi info**
Da una prima analisi della info si nota che c'è una colonna senza nome che, non sapendo a cosa si riferisce, va analizzato il contenuto per eventualmente dropparla. Essendo molte le righe con valuri nulli, si analizzerà l'utilità della colonna per poi decidere come usarla, mentre per le righe con valori nulli di happiness score, considerando che sarà usata dall'algoritmo di ML, si sostituiranno quanti più valori nulli possibili

In [82]:
print(df["Unnamed: 0"])

0          0
1          1
2          2
3          3
4          4
        ... 
1226    1226
1227    1227
1228    1228
1229    1229
1230    1230
Name: Unnamed: 0, Length: 1231, dtype: int64


In [83]:
#la colonna unnamed contiene il conteggio delle colonne e potrà essere eliminata
#inoltre region ed happiness rank sono ininfluenti per l'analisi
#standard error potrebbe essere utile, ma ha troppi valori nulli
df = df.drop(columns=["Unnamed: 0", "Region", "Happiness Rank", "Standard Error"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        617 non-null    object 
 1   Happiness Score                315 non-null    float64
 2   Economy (GDP per Capita)       315 non-null    float64
 3   Family                         470 non-null    float64
 4   Health (Life Expectancy)       315 non-null    float64
 5   Freedom                        470 non-null    float64
 6   Trust (Government Corruption)  315 non-null    float64
 7   Generosity                     1084 non-null   float64
 8   Dystopia Residual              315 non-null    float64
 9   year                           1231 non-null   int64  
dtypes: float64(8), int64(1), object(1)
memory usage: 96.3+ KB


In [84]:
#Gestisco l'happiness score aggiungendo la media e la varianza ai valori mancanti per avere varietà di valori

indici_null = df[df['Happiness Score'].isna()].index

for i, idx in enumerate(indici_null):
    media = df['Happiness Score'].mean()
    mediana = df['Happiness Score'].median()    
    if i % 2 == 0:  
        df.at[idx, 'Happiness Score'] = media
    else:             
        df.at[idx, 'Happiness Score'] = mediana
        
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        617 non-null    object 
 1   Happiness Score                1231 non-null   float64
 2   Economy (GDP per Capita)       315 non-null    float64
 3   Family                         470 non-null    float64
 4   Health (Life Expectancy)       315 non-null    float64
 5   Freedom                        470 non-null    float64
 6   Trust (Government Corruption)  315 non-null    float64
 7   Generosity                     1084 non-null   float64
 8   Dystopia Residual              315 non-null    float64
 9   year                           1231 non-null   int64  
dtypes: float64(8), int64(1), object(1)
memory usage: 96.3+ KB


In [85]:
#utilizzo lo stesso metodo dell'happiness score per gli altri valori nulli
indici_null = df[df['Economy (GDP per Capita)'].isna()].index

for i, idx in enumerate(indici_null):
    media = df['Economy (GDP per Capita)'].mean()
    mediana = df['Economy (GDP per Capita)'].median()    
    if i % 2 == 0:  
        df.at[idx, 'Economy (GDP per Capita)'] = media
    else:             
        df.at[idx, 'Economy (GDP per Capita)'] = mediana    
        
indici_null = df[df['Family'].isna()].index

for i, idx in enumerate(indici_null):
    media = df['Family'].mean()
    mediana = df['Family'].median()    
    if i % 2 == 0:  
        df.at[idx, 'Family'] = media
    else:             
        df.at[idx, 'Family'] = mediana     

indici_null = df[df['Health (Life Expectancy)'].isna()].index

for i, idx in enumerate(indici_null):
    media = df['Health (Life Expectancy)'].mean()
    mediana = df['Health (Life Expectancy)'].median()    
    if i % 2 == 0:  
        df.at[idx, 'Health (Life Expectancy)'] = media
    else:             
        df.at[idx, 'Health (Life Expectancy)'] = mediana  
        
indici_null = df[df['Freedom'].isna()].index

for i, idx in enumerate(indici_null):
    media = df['Freedom'].mean()
    mediana = df['Freedom'].median()    
    if i % 2 == 0:  
        df.at[idx, 'Freedom'] = media
    else:             
        df.at[idx, 'Freedom'] = mediana 
        
indici_null = df[df['Trust (Government Corruption)'].isna()].index

for i, idx in enumerate(indici_null):
    media = df['Trust (Government Corruption)'].mean()
    mediana = df['Trust (Government Corruption)'].median()    
    if i % 2 == 0:  
        df.at[idx, 'Trust (Government Corruption)'] = media
    else:             
        df.at[idx, 'Trust (Government Corruption)'] = mediana 
        
indici_null = df[df['Generosity'].isna()].index

for i, idx in enumerate(indici_null):
    media = df['Generosity'].mean()
    mediana = df['Generosity'].median()    
    if i % 2 == 0:  
        df.at[idx, 'Generosity'] = media
    else:             
        df.at[idx, 'Generosity'] = mediana
        
indici_null = df[df['Dystopia Residual'].isna()].index

for i, idx in enumerate(indici_null):
    media = df['Dystopia Residual'].mean()
    mediana = df['Dystopia Residual'].median()    
    if i % 2 == 0:  
        df.at[idx, 'Dystopia Residual'] = media
    else:             
        df.at[idx, 'Dystopia Residual'] = mediana
        
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        617 non-null    object 
 1   Happiness Score                1231 non-null   float64
 2   Economy (GDP per Capita)       1231 non-null   float64
 3   Family                         1231 non-null   float64
 4   Health (Life Expectancy)       1231 non-null   float64
 5   Freedom                        1231 non-null   float64
 6   Trust (Government Corruption)  1231 non-null   float64
 7   Generosity                     1231 non-null   float64
 8   Dystopia Residual              1231 non-null   float64
 9   year                           1231 non-null   int64  
dtypes: float64(8), int64(1), object(1)
memory usage: 96.3+ KB


In [86]:
#vedo i valori unici di ogni colonna con dati mancanti
print(df["Country"].value_counts())

Country
Switzerland          4
Jamaica              4
Honduras             4
Hungary              4
Lebanon              4
                    ..
Somaliland region    1
Puerto Rico          1
Swaziland            1
Djibouti             1
xx                   1
Name: count, Length: 192, dtype: int64


In [87]:
#rimuovo le righe che hanno il valore xx
df = df.drop(df[df["Country"] == "xx"].index)

#clean
df["Country"] = df["Country"].astype(str)

#rimozione simboli finali tipo * ^ †
df["Country"] = df["Country"].str.replace(r"[^A-Za-z\s]+$", "", regex=True)

#rimozione eventuali simboli strani interni
df["Country"] = df["Country"].str.replace(r"[^A-Za-z\s]", "", regex=True)

#normalizza spazi
df["Country"] = df["Country"].str.strip()
df["Country"] = df["Country"].str.replace(r"\s+", " ", regex=True)

#title case
df["Country"] = df["Country"].str.title()

print(df["Country"].value_counts())

Country
Nan                       614
Switzerland                 4
Tajikistan                  4
Dominican Republic          4
Mongolia                    4
                         ... 
Hong Kong Sar Of China      1
North Macedonia             1
Gambia                      1
Congo                       1
Eswatini Kingdom Of         1
Name: count, Length: 172, dtype: int64


In [ ]:
df = df.drop(df[df["Country"] == "Eswatini Kingdom Of"].index)

In [88]:
#valutazione media e deviazione standard
df.describe()

,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual,year
count,1230.000000,1230.000000,1230.000000,1230.000000,1230.000000,1230.000000,1230.000000,1230.000000,1230.000000
mean,5.335544,0.932938,1.003599,0.616898,0.408722,0.123557,0.154062,2.212029,2018.447154
std,0.578170,0.209516,0.197443,0.123238,0.093118,0.060120,0.157336,0.282416,2.282716
min,2.839000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.300907,0.328580,2015.000000
25%,5.291000,0.919754,0.995641,0.608244,0.405242,0.106130,0.080385,2.212029,2016.000000
50%,5.291000,0.966900,1.025070,0.640350,0.418272,0.106130,0.162000,2.212029,2018.000000
75%,5.353526,0.966900,1.025070,0.640350,0.418272,0.129460,0.239754,2.212029,2020.000000
max,7.587000,1.824270,1.610574,1.025250,0.669730,0.551910,0.838075,3.837720,2022.000000
